# RadioSSL Kaggle runbook (GPU)

This workflow combines real-patient Talisman patches with volume ROI and HRCT pilot patches, then applies a 75/25 GroupShuffleSplit by patient ID before SSL. Every source row for a patient stays in one partition. Patient -1 sentinel patches are appended to training only.


In [ ]:
# Kaggle notebook settings
REPO_URL = "https://github.com/Siddharthakhandelwal/o1.git"
from pathlib import Path
INPUT_ROOT = Path("/kaggle/input")
volume_roots = list(INPUT_ROOT.rglob("ILD_DB_volumeROIs"))
assert len(volume_roots) == 1, f"Expected one attached ILD dataset, found {len(volume_roots)} volume ROI folders"
DATASET_DIR = str(volume_roots[0].parent)
print("Using dataset:", DATASET_DIR)
REPO_DIR = "/kaggle/working/radiossl"
DATA_DIR = "/kaggle/working/data"
OUTPUT_DIR = "/kaggle/working/radiossl_run"

import os, torch
assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle settings.'
print(torch.cuda.get_device_name(0), f'{torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GiB VRAM')
assert os.path.isdir(os.path.join(DATASET_DIR, "ILD_DB_talismanTestSuite")), "Talisman folder missing from dataset root"

## 1. Clone the repository

In [ ]:
import subprocess
subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
%cd /kaggle/working/radiossl
!git rev-parse HEAD

## 2. Install dependencies (keep Kaggle CUDA PyTorch)

In [ ]:
# requirements-kaggle.txt should list project libraries, excluding torch/torchvision.
!python -m pip install -r requirements-kaggle.txt
!python -c "import torch, numpy, pydicom, tifffile, scipy, sklearn, skimage; print('imports ok; torch', torch.__version__, 'CUDA', torch.version.cuda)"

## 3. Extract and audit the original source data

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
!python scripts/audit_source_dataset.py --base_dir "{DATASET_DIR}" --output "{OUTPUT_DIR}/source_audit.json"

In [ ]:
import json
audit = json.load(open(f'{OUTPUT_DIR}/source_audit.json', encoding='utf-8'))
print({k: audit[k] for k in ['talisman_real_patient_id_count','volume_folder_id_count','hrct_pilot_case_count','volume_and_pilot_identifier_count','numeric_id_overlap_count','clinical_workbook_status','status']})
print('Use patient-level grouping; sentinel -1 is training-only.')

## 4. Build stride-32 patches and preserve labels/provenance

In [ ]:
!python step2_extract_and_package.py --base_dir "{DATASET_DIR}" --output_dir "{DATA_DIR}/processed" --stride 32
!python scripts/make_patient_split.py --data_path "{DATA_DIR}/processed/ild_dataset_combined.npz" --test_size 0.25 --seed 42 --patient_ids_same_namespace --output "{DATA_DIR}/patient_split.json"

## 5. Preflight: split checks and a few GPU steps
Confirm patient-disjoint assertions pass and the short run reports finite loss/gradients and peak VRAM before starting a long job.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
!python train_radiossl.py --data_path "{DATA_DIR}/processed/ild_dataset_combined.npz" --split_manifest "{DATA_DIR}/patient_split.json" --output_dir "{OUTPUT_DIR}/preflight" --epochs 1 --preflight_steps 3 --batch_size 16 --accum_steps 1 --amp

## 6. One full patient-aware P100 pretraining run
The 75/25 split is by unique real patient IDs, not patch rows. All rows from each ID stay together across sources. Sentinel -1 patches are excluded while choosing the split, then added to training. Test patients are excluded from SSL and radiomics scaling. A linear probe is logged every 10 epochs, including the held-out group split; the fixed epoch count does not select checkpoints.

In [ ]:
!python train_radiossl.py --data_path "{DATA_DIR}/processed/ild_dataset_combined.npz" --split_manifest "{DATA_DIR}/patient_split.json" --output_dir "{OUTPUT_DIR}/pretrain" --epochs 350 --batch_size 128 --accum_steps 2 --amp

## 7. Final probe, optional fine-tuning, and save artifacts

In [ ]:
!python scripts/evaluate_linear_probe.py --data_path "{DATA_DIR}/processed/ild_dataset_combined.npz" --split_manifest "{DATA_DIR}/patient_split.json" --checkpoint "{OUTPUT_DIR}/pretrain/radiossl_encoder_final.pth" --output "{OUTPUT_DIR}/linear_probe.json"
# Optional fine-tuning uses the same patient-disjoint manifest.
!cd /kaggle/working && zip -qr radiossl_run_artifacts.zip radiossl_run data/patient_split.json data/processed/dataset_summary.json